In [5]:
import os
import torch

num_layers = 36

# Root directory produced by dumping logic:
# {TENSOR_PATH}/layer_{layer_id}/q/{chunk_idx}.pt
# {TENSOR_PATH}/layer_{layer_id}/k/{chunk_idx}.pt
# {TENSOR_PATH}/layer_{layer_id}/v/{chunk_idx}.pt
TENSOR_PATH = "/data/jinda/kv-cache/Qwen3-4B-thinking-2507/qkv_test_10000-tokens"


def _list_chunk_ids(layer_dir: str):
    q_dir = os.path.join(layer_dir, "q")
    if not os.path.isdir(q_dir):
        return []

    chunk_ids = []
    for fname in os.listdir(q_dir):
        if fname.endswith(".pt"):
            stem = os.path.splitext(fname)[0]
            if stem.isdigit():
                chunk_ids.append(int(stem))
    return sorted(chunk_ids)


for layer_id in range(num_layers):
    layer_dir = os.path.join(TENSOR_PATH, f"layer_{layer_id}")
    chunk_ids = _list_chunk_ids(layer_dir)

    if len(chunk_ids) == 0:
        print(f"layer {layer_id}: no chunk files found, skip")
        continue

    tensor_q_list = []
    tensor_k_list = []
    tensor_v_list = []

    for chunk_idx in chunk_ids:
        tensor_name = f"{chunk_idx}.pt"
        q_path = os.path.join(layer_dir, "q", tensor_name)
        k_path = os.path.join(layer_dir, "k", tensor_name)
        v_path = os.path.join(layer_dir, "v", tensor_name)

        tensor_q_list.append(torch.load(q_path, map_location="cpu"))
        tensor_k_list.append(torch.load(k_path, map_location="cpu"))
        tensor_v_list.append(torch.load(v_path, map_location="cpu"))

    # Concat all chunks along token dimension -> full tensors [T, H, D]
    q_full = torch.cat(tensor_q_list, dim=0).contiguous()
    k_full = torch.cat(tensor_k_list, dim=0).contiguous()
    v_full = torch.cat(tensor_v_list, dim=0).contiguous()

    # TokenSize, HeadSize, DimSize
    print(f"layer: {layer_id}, q: {q_full.shape}, k: {k_full.shape}, v: {v_full.shape}")
    full_tensor = {
        "q": q_full,
        "k": k_full,
        "v": v_full,
    }

    save_path = os.path.join(layer_dir, "full_tensor.pt")
    torch.save(full_tensor, save_path)

    print(
        f"layer {layer_id}: saved {save_path}, "
        f"q={tuple(q_full.shape)}, k={tuple(k_full.shape)}, v={tuple(v_full.shape)}"
    )


layer: 0, q: torch.Size([10000, 32, 128]), k: torch.Size([10000, 8, 128]), v: torch.Size([10000, 8, 128])
layer 0: saved /data/jinda/kv-cache/Qwen3-4B-thinking-2507/qkv_test_10000-tokens/layer_0/full_tensor.pt, q=(10000, 32, 128), k=(10000, 8, 128), v=(10000, 8, 128)
layer: 1, q: torch.Size([10000, 32, 128]), k: torch.Size([10000, 8, 128]), v: torch.Size([10000, 8, 128])
layer 1: saved /data/jinda/kv-cache/Qwen3-4B-thinking-2507/qkv_test_10000-tokens/layer_1/full_tensor.pt, q=(10000, 32, 128), k=(10000, 8, 128), v=(10000, 8, 128)
layer: 2, q: torch.Size([10000, 32, 128]), k: torch.Size([10000, 8, 128]), v: torch.Size([10000, 8, 128])
layer 2: saved /data/jinda/kv-cache/Qwen3-4B-thinking-2507/qkv_test_10000-tokens/layer_2/full_tensor.pt, q=(10000, 32, 128), k=(10000, 8, 128), v=(10000, 8, 128)
layer: 3, q: torch.Size([10000, 32, 128]), k: torch.Size([10000, 8, 128]), v: torch.Size([10000, 8, 128])
layer 3: saved /data/jinda/kv-cache/Qwen3-4B-thinking-2507/qkv_test_10000-tokens/layer_3/f